In [2]:
from estimators import Delta, Delta_con, Gamma, Gamma_con, Gamma_con_Lap
from IMH import IndMH
import numpy.random as nr
import numpy as np
epoch = 100
Itr = 6000
N = 4096
d = 16
K = 40
S0 = 50
r = 0.05
v = 0.1
T = 1
dt = T/d
distr = "multivariate_normal"
objective = "Gamma"
conditional = False
ifLap = False

cmc_str = "F_CMC_" + objective
if conditional:
    obj = objective + '_con'
    if ifLap:
        obj = obj + '_Lap'
        d = d - 1
    print("===", obj, "Pricing problem = Independent Metropolis Hastings + control variate + conditional method ===")
else:
    obj = objective
    print("===", obj, "Pricing problem = Independent Metropolis Hastings + control variate ===")

est = eval(obj)(K, S0, r, v, T, d)
est.Generative_Matrix()
est.Lognormal_Approx()
if ifLap:
    est.Lap()
    mu1 = est.Lap_tar_mean
    L1 = est.Lap_tar_L
    mu2 = est.Lap_pro_mean
    L2 = est.Lap_pro_L
else:
    mu1 = est.default_mean
    L1 = est.default_L
    mu2 = est.proposal_mean
    L2 = est.proposal_L

=== Gamma Pricing problem = Independent Metropolis Hastings + control variate ===


In [ ]:
epoch = 500
cmc_mean = []

def CMC_Gamma(x):
    dx = est.default_mean + est.default_L @ x
    const = np.exp( - r * T) * K / (S0 ** 2 * v * np.sqrt(dt))
    Sa = np.mean(np.exp(dx))
    return const*x[0]*np.int64(Sa > K)
    

for i in range(epoch):
    data = nr.multivariate_normal(np.zeros(d), np.eye(d), Itr)
    data2 = np.apply_along_axis(CMC_Gamma, 1, data)
    cmc_mean.append(data2.mean())
    if (i + 1) % 10 == 0:
        print(i + 1, "replications completed !")

cmc_mean = np.array(cmc_mean)
print(cmc_mean.mean(), cmc_mean.var())

In [19]:
epoch = 1000
cmc_mean = []
imhcv_mean = []

def map_func(x):
    dx = est.default_mean + est.default_L @ x
    dy = est.proposal_mean + est.proposal_L @ x
    return dx,dy

for i in range(epoch):
    data = nr.multivariate_normal(np.zeros(d), np.eye(d), N)
    data2 = np.apply_along_axis(map_func, 1, data)
    datax = data2[:,0,:]
    datay = data2[:,1,:]
    imcv ,ma,cv,corf= est.CV(datax, datay)
    cmc = getattr(est, cmc_str)(datax)
    imhcv_mean.append(imcv)
    cmc_mean.append(cmc[0])
    if (i + 1) % 100 == 0:
        print(i + 1, "replications completed !")


cmc_mean = np.array(cmc_mean)
imhcv_mean = np.array(imhcv_mean)

VRF = cmc_mean.var()/imhcv_mean.var()
nl = len(str(round(VRF)))
print("=" * 10, "Compare with CMC", "=" * 10)
print(f"|    Mean of CMC    =  {cmc_mean.mean():.6f}      |")
print(f"|    Mean of CV     =  {imhcv_mean.mean():.6f}      |")
print(f"|    Var of CMC     =  {cmc_mean.var():.5e}   |")
print(f"|    Var of CV      =  {imhcv_mean.var():.5e}   |")
print("|    V R F          = ", round(VRF, 2), " " * (9 - nl), "|")
print('=' * 38)

100 replications completed !
200 replications completed !
300 replications completed !
400 replications completed !
500 replications completed !
600 replications completed !
700 replications completed !
800 replications completed !
900 replications completed !
1000 replications completed !
========== Compare with CMC ==========
|    Mean of CMC    =  -0.000182      |
|    Mean of CV     =  0.000041      |
|    Var of CMC     =  8.85599e-05   |
|    Var of CV      =  5.33732e-09   |
|    V R F          =  16592.57      |


In [3]:
epoch = 100
cmc_mean = []
imhcv_mean = []

    
for i in range(epoch):
    MyIMH = IndMH(Itr,N,dim = d,value=N+i)
    MyIMH.set_distribution_parameters(mu1, L1, mu2, L2)
    MyIMH.init_state()
    datax, datay, accept = MyIMH.IMH() 
    cmcdata = eval('nr.' + distr)(est.default_mean, est.default_L @ est.default_L.T, N)
    imcv = est.IMCV2(datax[-N:], datay[-N:], accept[-N:])
    cmc = getattr(est, cmc_str)(cmcdata)
    imhcv_mean.append(imcv[0])
    cmc_mean.append(cmc[0])
    if (i + 1) % 10 == 0:
        print(i + 1, "replications completed !")

cmc_mean = np.array(cmc_mean)
imhcv_mean = np.array(imhcv_mean)
VRF = cmc_mean.var()/imhcv_mean.var()
print("VRF:", VRF)

I0000 00:00:1696529620.752804    7673 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


VRF: 0.09696455662855655


In [4]:
nl = len(str(round(VRF)))
print("=" * 10, "Compare with CMC", "=" * 10)
print(f"|    Mean of CMC    =  {cmc_mean.mean():.6f}      |")
print(f"|    Mean of CV     =  {imhcv_mean.mean():.6f}      |")
print(f"|    Var of CMC     =  {cmc_mean.var():.5e}   |")
print(f"|    Var of CV      =  {imhcv_mean.var():.5e}   |")
print("|    V R F          = ", round(VRF, 2), " " * (9 - nl), "|")
print('=' * 38)

========== Compare with CMC ==========
|    Mean of CMC    =  0.000238      |
|    Mean of CV     =  -0.000782      |
|    Var of CMC     =  1.01099e-04   |
|    Var of CV      =  1.04264e-03   |
|    V R F          =  0.1          |
